> **About this notebook**
>
> This notebook is part of the example gallery of [CLABTOOLKIT](https://github.com/connectomicslab/clabtoolkit), an open-source Python toolkit for routine processing, manipulation and visualization of neuroimaging data. It shows how to use the `pointstools` module.
>
> 📦 **Install:** `pip install clabtoolkit` ([PyPI](https://pypi.org/project/clabtoolkit/)) · 📖 **Docs:** [clabtoolkit.readthedocs.io](https://clabtoolkit.readthedocs.io) · 📄 **Paper:** [arXiv:2607.02638](https://arxiv.org/abs/2607.02638)
>
> **Author:** Yasser Alemán-Gómez (Connectomics Lab, Radiology Department, CHUV – Lausanne University Hospital)

## Testing Methods in the `POINTSTOOLS` Module

This Jupyter Notebook serves as a platform to rigorously test the various methods included in the Python module `POINTSTOOLS`.

This module is particularly valuable for researchers working with 3D point clouds: loading and saving
point coordinates, attaching scalar maps and colortables to the points, transforming and filtering them
by bounds or by attribute values, exporting them to tables, merging several point clouds and
visualizing them with the selected overlay.

The tests are organized into the following sections for clarity and ease of navigation:

### Table of Contents

---

* 🔷 1. [`PointCloud Class`](#pointcloud-class). Main class to represent and manipulate point clouds. Initialize a PointCloud object from point coordinates
    * 🔸 1.1. [`load`](#pointcloud-load). Load a point cloud from a file
    * 🔸 1.2. [`save`](#pointcloud-save). Save the point cloud to a file
    * 🔸 1.3. [`copy`](#pointcloud-copy). Create a deep copy of the PointCloud object
    * 🔸 1.4. [`add_point_data`](#pointcloud-add_point_data). Add scalar data associated with each point
    * 🔸 1.5. [`load_colortable`](#pointcloud-load_colortable). Load a colortable from a file and associate it with a specified map name
    * 🔸 1.6. [`list_maps`](#pointcloud-list_maps). List all the available scalar maps in the point cloud
    * 🔸 1.7. [`apply_affine`](#pointcloud-apply_affine). Apply an affine transformation to the point coordinates
    * 🔸 1.8. [`filter_by_bounds`](#pointcloud-filter_by_bounds). Filter the points based on spatial bounds
    * 🔸 1.9. [`filter`](#pointcloud-filter). Filter the points based on coordinate values or point_data attributes
    * 🔸 1.10. [`get_bounds`](#pointcloud-get_bounds). Compute the bounding box of the point cloud
    * 🔸 1.11. [`get_centroid`](#pointcloud-get_centroid). Compute the centroid (geometric center) of the point cloud
    * 🔸 1.12. [`append`](#pointcloud-append). Append another PointCloud to this one by concatenating coordinates and data
    * 🔸 1.13. [`to_dataframe`](#pointcloud-to_dataframe). Convert the point cloud to a pandas DataFrame
    * 🔸 1.14. [`get_info`](#pointcloud-get_info). Display comprehensive information about the point cloud
    * 🔸 1.15. [`get_pointwise_colors`](#pointcloud-get_pointwise_colors). Compute the point colors for visualization based on the specified overlay
    * 🔸 1.16. [`plot`](#pointcloud-plot). Plot the point cloud with the specified overlay and visualization parameters

* 🔷 2. [`Other Methods`](#points-other_methods). Other methods to manipulate point clouds outside the PointCloud class
    * 🔸 2.1. [`merge_pointclouds`](#points-merge_pointclouds). Merge multiple point clouds into a single point cloud
    * 🔸 2.2. [`smooth_curve_coordinates`](#points-smooth_curve_coordinates). Smooth a 3D curve using Gaussian-weighted neighborhood averaging

---

---
## <a id="pointcloud-class"></a>1. PointCloud Class
Main class to represent and manipulate point clouds. Initialize a PointCloud object from point coordinates.

### 1.1 `load`
<a id="pointcloud-load"></a>

Load a point cloud from a file.

In [ ]:
########################### Testing load ###########################
import clabtoolkit.pointstools as cltpts
import clabtoolkit.parcellationtools as cltparc
import numpy as np
import pandas as pd
import os

# Point cloud with the centroids of a simulated parcellation (one color and one name per region)
parc = cltparc.Parcellation.simulate_parcellation(n_regions=20, seed=42)
cent_df = parc.compute_centroids()
pc = cltpts.PointCloud(
    points=cent_df[["x_mm", "y_mm", "z_mm"]].to_numpy(),
    region_colors=cent_df["color"].tolist(),
    region_names=cent_df["name"].tolist(),
    name="centroids",
)
pc.add_point_data(cent_df["volume"].to_numpy(), name="volume")

os.makedirs("/tmp/pointstools_examples", exist_ok=True)

print("Test 1: Loading a point cloud saved in npy format (coordinates, affine, name and point data)...")
pc.save("/tmp/pointstools_examples/centroids.npy")
pc_npy = cltpts.PointCloud.load("/tmp/pointstools_examples/centroids.npy")
print(f"  {pc_npy}")
print(f"  Maps: {pc_npy.list_maps()}")
print("-------------------------------")
print(" ")

print("Test 2: Loading a point cloud from a CSV file. The columns other than X, Y and Z become point data...")
pc.save("/tmp/pointstools_examples/centroids.csv", format="csv")
pc_csv = cltpts.PointCloud.load("/tmp/pointstools_examples/centroids.csv", format="csv")
print(f"  {pc_csv}")
print(f"  Maps: {pc_csv.list_maps()}")
print("-------------------------------")
print(" ")

print("Test 3: Loading a point cloud from a tab-separated text file written by another program...")
table = pd.DataFrame({"X": [10.0, -12.5, 3.2], "Y": [5.1, 8.0, -20.4], "Z": [0.0, 14.3, 7.7], "fa": [0.45, 0.61, 0.38]})
table.to_csv("/tmp/pointstools_examples/seeds.txt", sep="\t", index=False)
pc_txt = cltpts.PointCloud.load("/tmp/pointstools_examples/seeds.txt", format="txt")
print(f"  {pc_txt}")
print(f"  FA values: {pc_txt.point_data['fa']}")
print("-------------------------------")

### 1.2 `save`
<a id="pointcloud-save"></a>

Save the point cloud to a file.

In [ ]:
########################### Testing save ###########################
import clabtoolkit.pointstools as cltpts
import clabtoolkit.parcellationtools as cltparc
import numpy as np
import pandas as pd
import os

# Point cloud with the centroids of a simulated parcellation (one color and one name per region)
parc = cltparc.Parcellation.simulate_parcellation(n_regions=20, seed=42)
cent_df = parc.compute_centroids()
pc = cltpts.PointCloud(
    points=cent_df[["x_mm", "y_mm", "z_mm"]].to_numpy(),
    region_colors=cent_df["color"].tolist(),
    region_names=cent_df["name"].tolist(),
    name="centroids",
)
pc.add_point_data(cent_df["volume"].to_numpy(), name="volume")

pc.plot(maps=["volume"], radius=30)

os.makedirs("/tmp/pointstools_examples", exist_ok=True)

print("Test 1: Saving the point cloud in npy format (keeps the coordinates, affine, name and all the maps)...")
pc.save("/tmp/pointstools_examples/centroids.npy")
print(f"  Saved: {os.listdir('/tmp/pointstools_examples')}")
print("-------------------------------")
print(" ")

print("Test 2: Saving the point cloud as a CSV table including the colortable (index, name and color columns)...")
pc.save("/tmp/pointstools_examples/centroids_ctab.csv", format="csv", include_colortable=True)
print(pd.read_csv("/tmp/pointstools_examples/centroids_ctab.csv").head(3))
print("-------------------------------")
print(" ")

print("Test 3: Saving the point cloud as a tab-separated text file...")
pc.save("/tmp/pointstools_examples/centroids.txt", format="txt")
with open("/tmp/pointstools_examples/centroids.txt") as f:
    print("".join(f.readlines()[:3]))
print("-------------------------------")

### 1.3 `copy`
<a id="pointcloud-copy"></a>

Create a deep copy of the PointCloud object.

In [ ]:
########################### Testing copy ###########################
import clabtoolkit.pointstools as cltpts
import clabtoolkit.parcellationtools as cltparc
import numpy as np

# Point cloud with the centroids of a simulated parcellation (one color and one name per region)
parc = cltparc.Parcellation.simulate_parcellation(n_regions=20, seed=42)
cent_df = parc.compute_centroids()
pc = cltpts.PointCloud(
    points=cent_df[["x_mm", "y_mm", "z_mm"]].to_numpy(),
    region_colors=cent_df["color"].tolist(),
    region_names=cent_df["name"].tolist(),
    name="centroids",
)
pc.add_point_data(cent_df["volume"].to_numpy(), name="volume")

print("Test 1: Creating an independent copy of the point cloud...")
pc_copy = pc.copy()
pc_copy.coords[:, 0] += 100            # Move the copy 100 mm along X
pc_copy.name = "centroids_moved"
print(f"  Original: {pc.name:16s} first point: {np.round(pc.coords[0], 2)}")
print(f"  Copy    : {pc_copy.name:16s} first point: {np.round(pc_copy.coords[0], 2)}")
print("-------------------------------")

### 1.4 `add_point_data`
<a id="pointcloud-add_point_data"></a>

Add scalar data associated with each point.

In [ ]:
########################### Testing add_point_data ###########################
import clabtoolkit.pointstools as cltpts
import clabtoolkit.parcellationtools as cltparc
import numpy as np

# Point cloud with the centroids of a simulated parcellation (one color and one name per region)
parc = cltparc.Parcellation.simulate_parcellation(n_regions=20, seed=42)
cent_df = parc.compute_centroids()
pc = cltpts.PointCloud(
    points=cent_df[["x_mm", "y_mm", "z_mm"]].to_numpy(),
    region_colors=cent_df["color"].tolist(),
    region_names=cent_df["name"].tolist(),
    name="centroids",
)
pc.add_point_data(cent_df["volume"].to_numpy(), name="volume")

print("Test 1: Adding a scalar map (distance of each centroid to the center of the point cloud)...")
distance = np.linalg.norm(pc.coords - pc.get_centroid(), axis=1)
pc.add_point_data(distance, name="distance")
print(f"  Maps: {pc.list_maps()}")
print(f"  Distances: {np.round(pc.point_data['distance'][:5], 2)} ...")
print("-------------------------------")
print(" ")

print("Test 2: Adding a map with a specific data type (hemisphere labels as integers)...")
hemi = np.where(pc.coords[:, 0] < 0, 1, 2)       # 1 = left, 2 = right
pc.add_point_data(hemi, name="hemisphere", dtype=np.int16)
print(f"  Data type: {pc.point_data['hemisphere'].dtype}, values: {np.unique(pc.point_data['hemisphere'])}")
print("-------------------------------")
print(" ")

print("Test 3: Adding a map with a wrong number of values...")
try:
    pc.add_point_data(np.zeros(5), name="wrong")
except ValueError as e:
    print(f"  ValueError: {e}")
print("-------------------------------")

### 1.5 `load_colortable`
<a id="pointcloud-load_colortable"></a>

Load a colortable from a file and associate it with a specified map name.

In [ ]:
########################### Testing load_colortable ###########################
import clabtoolkit.pointstools as cltpts
import clabtoolkit.parcellationtools as cltparc
import numpy as np
import os

# Point cloud with the centroids of a simulated parcellation (one color and one name per region)
parc = cltparc.Parcellation.simulate_parcellation(n_regions=20, seed=42)
cent_df = parc.compute_centroids()
pc = cltpts.PointCloud(
    points=cent_df[["x_mm", "y_mm", "z_mm"]].to_numpy(),
    region_colors=cent_df["color"].tolist(),
    region_names=cent_df["name"].tolist(),
    name="centroids",
)
pc.add_point_data(cent_df["volume"].to_numpy(), name="volume")

os.makedirs("/tmp/pointstools_examples", exist_ok=True)

# Hemisphere labels and a lookup table (FreeSurfer LUT format: index name R G B A)
pc.add_point_data(np.where(pc.coords[:, 0] < 0, 1, 2), name="hemisphere")
lut_file = "/tmp/pointstools_examples/hemispheres.lut"
with open(lut_file, "w") as f:
    f.write("1  Left-Hemisphere   220  20  60  0\n")
    f.write("2  Right-Hemisphere   30 144 255  0\n")

print("Test 1: Loading a lookup table and associating it with the hemisphere map...")
pc.load_colortable(lut_file, map_name="hemisphere")
print(f"  Colortables: {list(pc.colortables.keys())}")
print(f"  Names      : {pc.colortables['hemisphere']['names']}")
print(f"  Colors     :\n{pc.colortables['hemisphere']['color_table']}")
print("-------------------------------")
print(" ")

# Changing the radius of the points in the plot
pc.plot(maps=["hemisphere"], radius=30)    

print("Test 2: Loading the same lookup table with a different opacity for each entry...")
pc.load_colortable(lut_file, map_name="hemisphere", opacity=np.array([1.0, 0.3]))
print(f"  Opacities: {pc.colortables['hemisphere']['color_table'][:, 3]}")
print("-------------------------------")

### 1.6 `list_maps`
<a id="pointcloud-list_maps"></a>

List all the available scalar maps in the point cloud.

In [ ]:
########################### Testing list_maps ###########################
import clabtoolkit.pointstools as cltpts
import clabtoolkit.parcellationtools as cltparc
import numpy as np

# Point cloud with the centroids of a simulated parcellation (one color and one name per region)
parc = cltparc.Parcellation.simulate_parcellation(n_regions=20, seed=42)
cent_df = parc.compute_centroids()
pc = cltpts.PointCloud(
    points=cent_df[["x_mm", "y_mm", "z_mm"]].to_numpy(),
    region_colors=cent_df["color"].tolist(),
    region_names=cent_df["name"].tolist(),
    name="centroids",
)
pc.add_point_data(cent_df["volume"].to_numpy(), name="volume")

print("Test 1: Listing the maps of the point cloud...")
print("NOTE: The 'default' map is created by the constructor and stores the region of each point.")
print(f"  Maps: {pc.list_maps()}")
print("-------------------------------")
print(" ")

print("Test 2: Listing the maps after adding a new one...")
pc.add_point_data(np.abs(pc.coords[:, 2]), name="abs_z")
print(f"  Maps: {pc.list_maps()}")
print("-------------------------------")

### 1.7 `apply_affine`
<a id="pointcloud-apply_affine"></a>

Apply an affine transformation to the point coordinates.

In [ ]:
########################### Testing apply_affine ###########################
import clabtoolkit.pointstools as cltpts
import clabtoolkit.parcellationtools as cltparc
import numpy as np

# Point cloud with the centroids of a simulated parcellation (one color and one name per region)
parc = cltparc.Parcellation.simulate_parcellation(n_regions=20, seed=42)
cent_df = parc.compute_centroids()
pc = cltpts.PointCloud(
    points=cent_df[["x_mm", "y_mm", "z_mm"]].to_numpy(),
    region_colors=cent_df["color"].tolist(),
    region_names=cent_df["name"].tolist(),
    name="centroids",
)
pc.add_point_data(cent_df["volume"].to_numpy(), name="volume")

# Translation of 10 mm along X and rotation of 90 degrees around Z
translation = np.eye(4)
translation[0, 3] = 10
rotation = np.array([[0, -1, 0, 0],
                     [1,  0, 0, 0],
                     [0,  0, 1, 0],
                     [0,  0, 0, 1]], dtype=float)

print("Test 1: Translating the point cloud 10 mm along X (in place)...")
before = pc.coords[0].copy()
pc.apply_affine(translation)
print(f"  First point before: {np.round(before, 2)}")
print(f"  First point after : {np.round(pc.coords[0], 2)}")
print("-------------------------------")
print(" ")

print("Test 2: Undoing the translation with the inverse of the affine...")
pc.apply_affine(translation, inverse=True)
print(f"  First point: {np.round(pc.coords[0], 2)}")
print(f"  Accumulated affine is the identity: {np.allclose(pc.affine, np.eye(4))}")
print("-------------------------------")
print(" ")

print("Test 3: Rotating a copy of the point cloud (inplace=False) and keeping the original untouched...")
pc_rot = pc.apply_affine(rotation, inplace=False)
print(f"  Original first point: {np.round(pc.coords[0], 2)}")
print(f"  Rotated first point : {np.round(pc_rot.coords[0], 2)}")
print("-------------------------------")

### 1.8 `filter_by_bounds`
<a id="pointcloud-filter_by_bounds"></a>

Filter the points based on spatial bounds.

In [ ]:
########################### Testing filter_by_bounds ###########################
import clabtoolkit.pointstools as cltpts
import clabtoolkit.parcellationtools as cltparc
import numpy as np

# Point cloud with the centroids of a simulated parcellation (one color and one name per region)
parc = cltparc.Parcellation.simulate_parcellation(n_regions=20, seed=42)
cent_df = parc.compute_centroids()
pc = cltpts.PointCloud(
    points=cent_df[["x_mm", "y_mm", "z_mm"]].to_numpy(),
    region_colors=cent_df["color"].tolist(),
    region_names=cent_df["name"].tolist(),
    name="centroids",
)
pc.add_point_data(cent_df["volume"].to_numpy(), name="volume")

print("Test 1: Keeping the centroids of the left hemisphere (X < 0)...")
pc_left = pc.filter_by_bounds(x_range=(-np.inf, 0), inplace=False)
print(f"  {len(pc)} points -> {len(pc_left)} points")
print("-------------------------------")
print(" ")

print("Test 2: Keeping the centroids inside a box and checking that the maps are filtered too...")
pc.filter_by_bounds(x_range=(-40, 40), y_range=(-40, 40), z_range=(-30, 30))
print(f"  Points: {len(pc)}, values in the volume map: {len(pc.point_data['volume'])}")
print(f"  Bounds: { {k: (round(float(v[0]), 1), round(float(v[1]), 1)) for k, v in pc.get_bounds().items()} }")
print("-------------------------------")

### 1.9 `filter`
<a id="pointcloud-filter"></a>

Filter the points based on coordinate values or point_data attributes.

In [ ]:
########################### Testing filter ###########################
import clabtoolkit.pointstools as cltpts
import clabtoolkit.parcellationtools as cltparc
import numpy as np

# Point cloud with the centroids of a simulated parcellation (one color and one name per region)
parc = cltparc.Parcellation.simulate_parcellation(n_regions=20, seed=42)
cent_df = parc.compute_centroids()
pc = cltpts.PointCloud(
    points=cent_df[["x_mm", "y_mm", "z_mm"]].to_numpy(),
    region_colors=cent_df["color"].tolist(),
    region_names=cent_df["name"].tolist(),
    name="centroids",
)
pc.add_point_data(cent_df["volume"].to_numpy(), name="volume")

print("Test 1: Keeping the centroids above the axial plane Z = 0...")
pc_sup = pc.filter("Z > 0", inplace=False)
print("-------------------------------")
print(" ")

print("Test 2: Keeping the largest regions using a map of the point cloud (volume)...")
pc_large = pc.filter(f"volume >= {np.median(pc.point_data['volume'])}", inplace=False)
print(f"  Names of the kept regions: {pc_large.colortables['default']['names'][:5]} ...")
print("-------------------------------")
print(" ")

print("Test 3: Keeping the centroids within a range of values of a map...")
pc.add_point_data(np.random.default_rng(42).random(len(pc)), name="intensity")
pc_mid = pc.filter("0.25 <= intensity <= 0.75", inplace=False)
print(f"  Intensity range of the kept points: {pc_mid.point_data['intensity'].min():.2f} - {pc_mid.point_data['intensity'].max():.2f}")
print("-------------------------------")

### 1.10 `get_bounds`
<a id="pointcloud-get_bounds"></a>

Compute the bounding box of the point cloud.

In [ ]:
########################### Testing get_bounds ###########################
import clabtoolkit.pointstools as cltpts
import clabtoolkit.parcellationtools as cltparc
import numpy as np

# Point cloud with the centroids of a simulated parcellation (one color and one name per region)
parc = cltparc.Parcellation.simulate_parcellation(n_regions=20, seed=42)
cent_df = parc.compute_centroids()
pc = cltpts.PointCloud(
    points=cent_df[["x_mm", "y_mm", "z_mm"]].to_numpy(),
    region_colors=cent_df["color"].tolist(),
    region_names=cent_df["name"].tolist(),
    name="centroids",
)
pc.add_point_data(cent_df["volume"].to_numpy(), name="volume")

print("Test 1: Getting the bounding box of the point cloud...")
bounds = pc.get_bounds()
for axis, (vmin, vmax) in bounds.items():
    print(f"  {axis.upper()}: {vmin:7.2f} to {vmax:7.2f} mm (size: {vmax - vmin:.2f} mm)")
print("-------------------------------")
print(" ")

print("Test 2: Getting the bounding box of an empty point cloud...")
print(f"  {cltpts.PointCloud().get_bounds()}")
print("-------------------------------")

### 1.11 `get_centroid`
<a id="pointcloud-get_centroid"></a>

Compute the centroid (geometric center) of the point cloud.

In [ ]:
########################### Testing get_centroid ###########################
import clabtoolkit.pointstools as cltpts
import clabtoolkit.parcellationtools as cltparc
import numpy as np

# Point cloud with the centroids of a simulated parcellation (one color and one name per region)
parc = cltparc.Parcellation.simulate_parcellation(n_regions=20, seed=42)
cent_df = parc.compute_centroids()
pc = cltpts.PointCloud(
    points=cent_df[["x_mm", "y_mm", "z_mm"]].to_numpy(),
    region_colors=cent_df["color"].tolist(),
    region_names=cent_df["name"].tolist(),
    name="centroids",
)
pc.add_point_data(cent_df["volume"].to_numpy(), name="volume")

print("Test 1: Getting the centroid (geometric center) of the point cloud...")
center = pc.get_centroid()
print(f"  Centroid: {np.round(center, 2)}")
print("-------------------------------")
print(" ")

print("Test 2: Centering the point cloud at the origin using its centroid...")
centering = np.eye(4)
centering[:3, 3] = -center
pc.apply_affine(centering)
print(f"  Centroid after centering: {np.round(pc.get_centroid(), 6)}")
print("-------------------------------")

### 1.12 `append`
<a id="pointcloud-append"></a>

Append another PointCloud to this one by concatenating coordinates and data.

In [ ]:
########################### Testing append ###########################
import clabtoolkit.pointstools as cltpts
import clabtoolkit.parcellationtools as cltparc
import numpy as np

# Point cloud with the centroids of a simulated parcellation (one color and one name per region)
parc = cltparc.Parcellation.simulate_parcellation(n_regions=20, seed=42)
cent_df = parc.compute_centroids()
pc = cltpts.PointCloud(
    points=cent_df[["x_mm", "y_mm", "z_mm"]].to_numpy(),
    region_colors=cent_df["color"].tolist(),
    region_names=cent_df["name"].tolist(),
    name="centroids",
)
pc.add_point_data(cent_df["volume"].to_numpy(), name="volume")

import warnings

# A second point cloud with 5 random seeds and a different map
seeds = cltpts.PointCloud(points=np.random.default_rng(0).normal(0, 20, (5, 3)), name="seeds")
seeds.add_point_data(np.linspace(0.2, 0.6, 5), name="fa")

print("Test 1: Appending the seeds to a copy of the centroids (inplace)...")
pc_all = pc.copy()
pc_all.append(seeds, handle_colortable_conflicts="skip")
print(f"  {len(pc)} + {len(seeds)} -> {pc_all}")
print("-------------------------------")
print(" ")

print("Test 2: Missing maps are filled with NaN (fill_value)...")
print(f"  volume map of the last 6 points: {pc_all.point_data['volume'][-6:]}")
print(f"  fa map of the first 3 points   : {pc_all.point_data['fa'][:3]}")
print("-------------------------------")
print(" ")

print("Test 3: Concatenating point clouds with the + operator (returns a new point cloud)...")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")         # Both point clouds have a 'default' colortable
    pc_sum = pc + seeds
print(f"  {pc_sum}")
print("-------------------------------")

### 1.13 `to_dataframe`
<a id="pointcloud-to_dataframe"></a>

Convert the point cloud to a pandas DataFrame.

In [ ]:
########################### Testing to_dataframe ###########################
import clabtoolkit.pointstools as cltpts
import clabtoolkit.parcellationtools as cltparc
import numpy as np

# Point cloud with the centroids of a simulated parcellation (one color and one name per region)
parc = cltparc.Parcellation.simulate_parcellation(n_regions=20, seed=42)
cent_df = parc.compute_centroids()
pc = cltpts.PointCloud(
    points=cent_df[["x_mm", "y_mm", "z_mm"]].to_numpy(),
    region_colors=cent_df["color"].tolist(),
    region_names=cent_df["name"].tolist(),
    name="centroids",
)
pc.add_point_data(cent_df["volume"].to_numpy(), name="volume")

print("Test 1: Converting the point cloud to a DataFrame (coordinates and maps)...")
print(pc.to_dataframe().head(3))
print("-------------------------------")
print(" ")

print("Test 2: Including the region index, name and color from the colortable...")
print(pc.to_dataframe(include_colortable=True).head(3))
print("-------------------------------")
print(" ")

print("Test 3: Getting only the coordinates...")
print(pc.to_dataframe(include_data=False).head(3))
print("-------------------------------")

### 1.14 `get_info`
<a id="pointcloud-get_info"></a>

Display comprehensive information about the point cloud.

In [ ]:
########################### Testing get_info ###########################
import clabtoolkit.pointstools as cltpts
import clabtoolkit.parcellationtools as cltparc
import numpy as np

# Point cloud with the centroids of a simulated parcellation (one color and one name per region)
parc = cltparc.Parcellation.simulate_parcellation(n_regions=20, seed=42)
cent_df = parc.compute_centroids()
pc = cltpts.PointCloud(
    points=cent_df[["x_mm", "y_mm", "z_mm"]].to_numpy(),
    region_colors=cent_df["color"].tolist(),
    region_names=cent_df["name"].tolist(),
    name="centroids",
)
pc.add_point_data(cent_df["volume"].to_numpy(), name="volume")

print("Test 1: Exploring the point cloud of region centroids...")
pc.get_info()
print("-------------------------------")

### 1.15 `get_pointwise_colors`
<a id="pointcloud-get_pointwise_colors"></a>

Compute the point colors for visualization based on the specified overlay.

In [ ]:
########################### Testing get_pointwise_colors ###########################
import clabtoolkit.pointstools as cltpts
import clabtoolkit.parcellationtools as cltparc
import numpy as np

# Point cloud with the centroids of a simulated parcellation (one color and one name per region)
parc = cltparc.Parcellation.simulate_parcellation(n_regions=20, seed=42)
cent_df = parc.compute_centroids()
pc = cltpts.PointCloud(
    points=cent_df[["x_mm", "y_mm", "z_mm"]].to_numpy(),
    region_colors=cent_df["color"].tolist(),
    region_names=cent_df["name"].tolist(),
    name="centroids",
)
pc.add_point_data(cent_df["volume"].to_numpy(), name="volume")

print("Test 1: Getting the colors of a map with a colortable (one color per region)...")
colors = pc.get_pointwise_colors("default")
print(f"  Shape: {colors.shape} (RGBA)")
print(f"  First region color: {np.round(colors[0], 3)} = {pc.to_dataframe(include_colortable=True)['color'][0]}")
print("-------------------------------")
print(" ")

print("Test 2: Getting the colors of a scalar map using a colormap...")
colors = pc.get_pointwise_colors("volume", colormap="hot")
print(f"  Shape: {colors.shape}")
print(f"  Smallest region: {np.round(colors[np.argmin(pc.point_data['volume'])], 3)}")
print(f"  Largest region : {np.round(colors[np.argmax(pc.point_data['volume'])], 3)}")
print("-------------------------------")
print(" ")

print("Test 3: Coloring in gray the values outside a range (range_min / range_max)...")
low, high = np.percentile(pc.point_data["volume"], [25, 75])
colors = pc.get_pointwise_colors("volume", colormap="viridis", range_min=low, range_max=high)
outside = (pc.point_data["volume"] < low) | (pc.point_data["volume"] > high)
print(f"  Points outside the range: {outside.sum()}, color: {np.round(colors[outside][0], 3)}")
print("-------------------------------")

### 1.16 `plot`
<a id="pointcloud-plot"></a>

Plot the point cloud with the specified overlay and visualization parameters.

In [ ]:
########################### Testing plot ###########################
import clabtoolkit.pointstools as cltpts
import clabtoolkit.parcellationtools as cltparc
import numpy as np
from IPython.display import Image, display
import os
os.makedirs('/tmp/pointstools_examples', exist_ok=True)

# Point cloud with the centroids of a simulated parcellation (one color and one name per region)
parc = cltparc.Parcellation.simulate_parcellation(n_regions=20, seed=42)
cent_df = parc.compute_centroids()
pc = cltpts.PointCloud(
    points=cent_df[["x_mm", "y_mm", "z_mm"]].to_numpy(),
    region_colors=cent_df["color"].tolist(),
    region_names=cent_df["name"].tolist(),
    name="centroids",
)
pc.add_point_data(cent_df["volume"].to_numpy(), name="volume")

print("Test 1: Plotting the volume of each region with a colormap and a colorbar...")
print("NOTE: save_path saves the figure. Without it, an interactive PyVista window is opened.")
pc.plot(maps="volume", cmap="hot", views=["lateral"], show_colorbar=True, colorbar_title="Volume (mm3)",
        save_path="/tmp/pointstools_examples/centroids_volume.png")
display(Image(filename="/tmp/pointstools_examples/centroids_volume.png", width=900))
print("-------------------------------")
print(" ")

print("Test 2: Plotting the regions with their own colors from two different views...")
pc.plot(maps="default", cmap=None, views=["lateral", "dorsal"],
        save_path="/tmp/pointstools_examples/centroids_regions.png")
display(Image(filename="/tmp/pointstools_examples/centroids_regions.png", width=900))
print("-------------------------------")

---
## <a id="points-other_methods"></a>2. Other Methods
Other methods to manipulate point clouds outside the PointCloud class.

### 2.1 `merge_pointclouds`
<a id="points-merge_pointclouds"></a>

Merge multiple point clouds into a single point cloud.

In [ ]:
########################### Testing merge_pointclouds ###########################
import clabtoolkit.pointstools as cltpts
import numpy as np
import os

rng = np.random.default_rng(42)
left = cltpts.PointCloud(points=rng.normal([-30, 0, 0], 8, (40, 3)), name="left_seeds")
right = cltpts.PointCloud(points=rng.normal([30, 0, 0], 8, (25, 3)), name="right_seeds")
right.add_point_data(rng.random(25), name="fa")

print("Test 1: Merging two point clouds. The ID of the source point cloud is stored in the 'point_id' map...")
merged = cltpts.merge_pointclouds([left, right])
print(f"  {merged}")
print(f"  Names : {merged.colortables['point_id']['names']}")
ids, counts = np.unique(merged.point_data["point_id"], return_counts=True)
print(f"  Points per ID: {dict(zip(ids.tolist(), counts.tolist()))}")
print("-------------------------------")
print(" ")

print("Test 2: Merging with a custom colortable and map name...")
ctable = {
    "names": ["Left", "Right"],
    "color_table": np.array([[0.86, 0.08, 0.24, 1.0, 1],
                             [0.12, 0.56, 1.00, 1.0, 2]]),
}
merged = cltpts.merge_pointclouds([left, right], color_table=ctable, map_name="hemisphere")
print(f"  Maps : {merged.list_maps()}")
print(f"  Color of the first point of each hemisphere:")
colors = merged.get_pointwise_colors("hemisphere")
print(f"    Left : {colors[0]}")
print(f"    Right: {colors[-1]}")
print("-------------------------------")
print(" ")

print("Test 3: Merging point clouds saved in npy files...")
os.makedirs("/tmp/pointstools_examples", exist_ok=True)
left.save("/tmp/pointstools_examples/left_seeds.npy")
right.save("/tmp/pointstools_examples/right_seeds.npy")
merged = cltpts.merge_pointclouds(["/tmp/pointstools_examples/left_seeds.npy",
                                   "/tmp/pointstools_examples/right_seeds.npy"])
print(f"  {merged}, names: {merged.colortables['point_id']['names']}")
print("-------------------------------")

### 2.2 `smooth_curve_coordinates`
<a id="points-smooth_curve_coordinates"></a>

Smooth a 3D curve using Gaussian-weighted neighborhood averaging.

In [ ]:
########################### Testing smooth_curve_coordinates ###########################
import clabtoolkit.pointstools as cltpts
import matplotlib.pyplot as plt
import numpy as np

# Noisy helix (e.g. a streamline or a trajectory)
t = np.linspace(0, 4 * np.pi, 120)
clean = np.column_stack([10 * np.cos(t), 10 * np.sin(t), 2 * t])
noisy = clean + np.random.default_rng(42).normal(0, 0.8, clean.shape)

print("Test 1: Smoothing a noisy 3D curve with the default parameters...")
smoothed = cltpts.smooth_curve_coordinates(noisy)
print(f"  Mean distance to the clean curve: noisy = {np.linalg.norm(noisy - clean, axis=1).mean():.3f}, "
      f"smoothed = {np.linalg.norm(smoothed - clean, axis=1).mean():.3f}")
print("-------------------------------")
print(" ")

print("Test 2: Comparing different amounts of smoothing...")
for sigma, iterations, window in [(1.0, 1, 5), (2.0, 3, 9), (4.0, 10, 15)]:
    sm = cltpts.smooth_curve_coordinates(noisy, sigma=sigma, iterations=iterations, window_size=window)
    error = np.linalg.norm(sm - clean, axis=1).mean()
    print(f"  sigma={sigma}, iterations={iterations:2d}, window_size={window:2d} -> mean distance: {error:.3f}")
print("NOTE: Too much smoothing shrinks the curve and moves it away from the clean one.")
print("-------------------------------")
print(" ")

print("Test 3: Plotting the noisy and the smoothed curves...")
smoothed = cltpts.smooth_curve_coordinates(noisy)
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(projection="3d")
ax.plot(*noisy.T, color="lightgray", label="Noisy")
ax.plot(*smoothed.T, color="crimson", linewidth=2, label="Smoothed")
ax.legend()
plt.show()
print("-------------------------------")